# Grid Sample Operator — Clean Experiment Notebook

This notebook contains the clean workflow for the `grid_sample` operator experiment.

The goal is to compare the PyTorch reference implementation with the Triton implementation, check correctness, tune kernel parameters, and summarize the final benchmark result.


## 1. Environment Check

First, check the PyTorch version, CUDA availability, GPU name, and Triton version.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available")

In [ ]:
# Install Triton if needed. In Colab this is usually already installed.
# In a local environment / Antigravity, you may not need to run this cell.
# !pip install triton

In [ ]:
import triton
print("Triton version:", triton.__version__)

## 2. Project Setup

Set the project folder that contains:

- `benchmark.py`
- `test_correctness.py`
- `reference.py`
- `triton_impl/`

In Colab, the project was located at:

```text
/content/drive/MyDrive/grid_sample
```

If running locally in Antigravity, place this notebook inside the `grid_sample` folder or change `PROJECT_DIR`.


In [ ]:
from pathlib import Path
import os

# Option 1: Colab path
colab_path = Path("/content/drive/MyDrive/grid_sample")

# Option 2: local path, useful in Antigravity/VS Code
local_path = Path.cwd()

if colab_path.exists():
    PROJECT_DIR = colab_path
else:
    PROJECT_DIR = local_path

os.chdir(PROJECT_DIR)
print("Current project directory:", PROJECT_DIR)
print("Files:")
for item in sorted(PROJECT_DIR.iterdir()):
    print("-", item.name)

## 3. Operator Overview

`grid_sample` samples values from an input image using coordinates from a grid.

Input shape:

```text
[N, C, H, W]
```

Grid shape:

```text
[N, H_out, W_out, 2]
```

Output shape:

```text
[N, C, H_out, W_out]
```

In this work, the supported configuration is:

- 4D image tensor
- `float32`
- CUDA tensor
- bilinear interpolation
- zero padding
- `align_corners=False`


## 4. Correctness Test

The Triton output is compared against PyTorch `torch.nn.functional.grid_sample`.

The result is accepted when the maximum absolute error is very small, around `1e-7`.


In [ ]:
!python test_correctness.py

## 5. Benchmark Test

Benchmark compares:

1. PyTorch reference
2. Triton `v1` original implementation
3. Tuned/best Triton implementation

The benchmark uses warmup iterations first, then timing iterations.


In [ ]:
!python benchmark.py

## 6. Final Benchmark Summary

The final clean benchmark result from the experiment was:

| Case | Shape | PyTorch mean (ms) | Triton best mean (ms) | Speedup vs PyTorch | Best Mode |
|---|---|---:|---:|---:|---|
| 1 | `1x32x64x64 -> 64x64` | 0.0666 | 0.1368 | 0.49x | `v1_original` |
| 2 | `4x64x128x128 -> 128x128` | 0.9884 | 0.5398 | 1.83x | `tuned_256_8` |
| 3 | `8x64x256x256 -> 256x256` | 18.9034 | 4.0825 | 4.63x | `tuned_128_4` |

Correctness result:

- Case 1 error: `2.38e-07`
- Case 2 error: `2.38e-07`
- Case 3 error: `4.77e-07`

All benchmark correctness checks passed.


## 7. Interpretation

For small input size, PyTorch is faster because the overhead of launching and running a custom Triton kernel is not worth it.

For medium and large input sizes, the Triton implementation becomes faster. The best result is shown in Case 3, where the tuned Triton kernel achieved about `4.63x` speedup over PyTorch.

The final implementation is correct because the output difference compared to PyTorch is very small, around `1e-7`.


## 8. Conclusion

The `grid_sample` Triton implementation successfully matches PyTorch correctness and improves performance for larger workloads.

Main result:

- Small case: slower than PyTorch
- Medium case: faster than PyTorch
- Large case: much faster than PyTorch

This shows that the custom Triton implementation is useful when the tensor size is large enough to benefit from GPU parallelism and tuning.
